## Model Testing

Testing how well this model performs as a standalone unit, procedures:

1. yFinance API used to gather current mixed-cap stock data
2. import the IVA model, feed the data, let it forecast valuations
3. prediction checks will be performed on July 2nd
4. real asset movement and prediction comparisons will be graphed
5. evaluation results will be added to the model card

#### Imports

In [1]:
import pandas as pd
import yfinance as yf
import joblib

#### Data Gathering

In [ ]:
# manual process, tickers need constant updates
# csv file may be different in the future or missing
data = []
tickers = ["JKHY", "FRT", "COP", "GLW", "UBER", "DE", "CRM", 
           "APH", "PFE", "ETN", "BX", "HON", "PLD", "CB", "VRT"]

for ticker in tickers:
    stock = yf.Ticker(ticker)
    info  = stock.info
    price = info.get("currentPrice")
    earnings = info.get("trailingEps")
    growth = info.get("earningsGrowth")
    pe_ratio = info.get("trailingPE")
    data.append({
        "ticker": ticker, "price": price,
        "earnings": earnings, "pe_ratio": pe_ratio,
        "growth": growth, "corp_yield": 2.65
    })

df = pd.DataFrame(data)
df.to_csv("testing_dataset.csv", index=False, mode='a', header=False)
print("Dataset created and saved")

Dataset created and saved


#### Updating Dataframe with Model Predictions

In [ ]:
df = pd.read_csv("testing_dataset.csv")
df.dropna(inplace=True)
df.drop_duplicates(subset=['ticker'], keep='first', inplace=True)

def calculate_intrinsic_value(row):
    cagr = row['growth'] * 100
    iv = (row['earnings'] * (8.5 + (2 * cagr)) * 4.4) / 2.67
    return float(round(iv, 2))

df['iv'] = df.apply(calculate_intrinsic_value, axis=1)
display(df.head())

,ticker,price,earnings,pe_ratio,growth,corp_yield,iv
0,NVDA,220.61,4.90,45.022450,0.956,2.65,1612.56
1,AAPL,298.97,8.25,36.238790,0.218,2.65,708.33
2,MSFT,417.42,16.77,24.890877,0.234,2.65,1528.27
3,AMZN,259.34,8.20,31.626830,0.748,2.65,2136.42
4,GOOGL,387.66,13.10,29.592365,0.820,2.65,3723.93


In [31]:
# loading the model and standard scaler used while training
# applying scaler to the features before feeding into the model
# crucial step since it scales current data to training conditions
artifacts = joblib.load('iv_analyzer.joblib')
scaler = artifacts['scaler']
nn_model = artifacts['model']

classifier_features = ['pe_ratio', 'iv', 'price']
x = df[classifier_features]
x_scaled = scaler.transform(x)

df['prediction'] = nn_model.predict(x_scaled)
probabilities = nn_model.predict_proba(x_scaled)
df['prob_for_1'] = probabilities[:, 1]
df['prob_for_0'] = probabilities[:, 0]
df.to_csv("testing_dataset.csv", index=False)
print("Model testing successful")

Model testing successful


#### Evaluating May 2026 Predictions Made by Model

The testing dataset with the May 2026 predictions will be loaded and each ticker's current price will also be queried from yFinance. The final evaluation will assume that 100 shares of each ticker from the testing dataset was considered theoretically for purchase within an investment account, if the model indicated (1) then the ticker was bought, otherwise it wasn't bought. Based on this theoretical assumption, the current price of each ticker, and the theoretically purchased shares, the value (lost or gained) of this theoretical investment account will be assessed, as an indication of the model's performance in real world settings.

In [5]:
# fetching current prices via yfinance
# by getting latest market price or regularMarketPrice
# then calculating gain/loss per share and per 100 shares
df = pd.read_csv("testing_dataset.csv")
current_prices = {}
for ticker in df['ticker']:
    try:
        stock = yf.Ticker(ticker)
        price = stock.info.get("currentPrice") or stock.info.get("regularMarketPrice")
        current_prices[ticker] = price
    except Exception:
        current_prices[ticker] = None
df['current_price'] = df['ticker'].map(current_prices)
df['price_change'] = df['current_price'] - df['price']
df['pct_change'] = (df['price_change'] / df['price']) * 100
df['pnl_100_shares'] = df['price_change'] * 100

In [6]:
# model decision evaluation where:
# buy = 1, sell = 0 as indicated within the testing dataset
def categorize_trade(row):
    if row['prediction'] == 1 and row['price_change'] > 0:
        return 'Profitable Buy'
    elif row['prediction'] == 1 and row['price_change'] <= 0:
        return 'Loss-Making Buy'
    elif row['prediction'] == 0 and row['price_change'] > 0:
        return 'Missed Gain (Skipped)'
    else:
        return 'Avoided Loss (Skipped)'

df['category'] = df.apply(categorize_trade, axis=1)

#### Summary of Evaluation Output

In [11]:
output_filename = "portfolio_summary.txt"
summary = df.groupby('category').agg(
    count=('ticker', 'count'),
    total_pnl=('pnl_100_shares', 'sum'),
    avg_pct_change=('pct_change', 'mean')
).reset_index()
loss_buys = df[df['category'] == 'Loss-Making Buy'].sort_values(by='pnl_100_shares')[
    ['ticker', 'price', 'current_price', 'price_change', 'pnl_100_shares']
]
profit_buys = df[df['category'] == 'Profitable Buy'].sort_values(by='pnl_100_shares', ascending=False)[
    ['ticker', 'price', 'current_price', 'price_change', 'pnl_100_shares']
]
missed_gains = df[df['category'] == 'Missed Gain (Skipped)'].sort_values(by='pnl_100_shares', ascending=False)[
    ['ticker', 'price', 'current_price', 'price_change', 'pnl_100_shares']
]
avoided_losses = df[df['category'] == 'Avoided Loss (Skipped)'].sort_values(by='pnl_100_shares')[
    ['ticker', 'price', 'current_price', 'price_change', 'pnl_100_shares']
]
with open(output_filename, 'w') as f:
    print("=== PORTFOLIO CATEGORY BREAKDOWN ===", file=f)
    print(summary.to_string(index=False), file=f)
    print("\n" + "="*50, file=f)
    print("1. LOSS-MAKING BUYS (Capital Lost)", file=f)
    print("="*50, file=f)
    print(loss_buys.to_string(index=False), file=f)
    print("\n" + "="*50, file=f)
    print("2. PROFITABLE BUYS (Capital Gained)", file=f)
    print("="*50, file=f)
    print(profit_buys.to_string(index=False), file=f)
    print("\n" + "="*50, file=f)
    print("3. MISSED GAINS (Skipped Stocks That Rose)", file=f)
    print("="*50, file=f)
    print(missed_gains.to_string(index=False), file=f)
    print("\n" + "="*50, file=f)
    print("4. AVOIDED LOSSES (Skipped Stocks That Fell)", file=f)
    print("="*50, file=f)
    print(avoided_losses.to_string(index=False), file=f)
print(f"Report successfully saved to '{output_filename}'")

Report successfully saved to 'portfolio_summary.txt'
